# Phase 1 — Data Engineering & EDA
## Blood Vessel Blockage Detection Project

In [ ]:
# Importing Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

---
## Step 1 — Load Dataset

In [ ]:
df = pd.read_csv('final.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe()

---
## Step 2 — Missing Value Analysis

**Why median imputation?**
- Medical data is often skewed
- Median is robust to outliers, mean is NOT
- Example: [1,2,3,4,100] → Mean=22 (wrong!), Median=3 (correct!)

In [ ]:
# Check missing values
df.isnull().sum()

In [ ]:
# Missing percentage
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_pct

In [ ]:
# Visualize missing values
plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Value Heatmap (Yellow = Missing)')
plt.tight_layout()
plt.show()

In [ ]:
# Missing % bar chart
plt.figure(figsize=(10, 4))
missing_pct.plot(kind='bar', color='tomato', edgecolor='black')
plt.title('Missing Value % Per Feature')
plt.ylabel('Missing %')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Apply Median Imputation
df = df.fillna(df.median(numeric_only=True))

# Verify no missing values remain
print('Missing values after imputation:', df.isnull().sum().sum())
df.head()

---
## Step 3 — Statistical Analysis

**Skewness Guide:**
- 0 = Perfect symmetric (Normal distribution)
- > 0.7 = Right skewed (tail on right side)
- < -0.7 = Left skewed (tail on left side)
- Skewed features hurt neural network training!

In [ ]:
df.describe()

In [ ]:
# Skewness of each feature
skewness = df.skew()
print('Skewness per feature:')
skewness

In [ ]:
# Visualize skewness
plt.figure(figsize=(10, 4))
colors = ['red' if abs(s) > 0.7 else 'green' for s in skewness]
skewness.plot(kind='bar', color=colors, edgecolor='black')
plt.axhline(0.7, color='red', linestyle='--', label='+0.7 threshold')
plt.axhline(-0.7, color='red', linestyle='--', label='-0.7 threshold')
plt.title('Feature Skewness (Red = Skewed, Green = Normal)')
plt.ylabel('Skewness Value')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Distribution plots for all features
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=8)
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5, label='Mean')
    axes[i].axvline(df[col].median(), color='green', linestyle='--', linewidth=1.5, label='Median')
    axes[i].legend(fontsize=6)

plt.suptitle('Feature Distributions — All Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 4 — Outlier Detection

**IQR Method:**
- IQR = Q3 - Q1
- Lower Fence = Q1 - 1.5 * IQR
- Upper Fence = Q3 + 1.5 * IQR
- Outside fence = Outlier

**Why Winsorize not Delete?**
- PSV = 160 cm/s might be REAL blockage data!
- Deleting = losing critical medical information
- Winsorize = cap at fence value, keep the record

In [ ]:
# Boxplots to visualize outliers
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    axes[i].boxplot(df[col], patch_artist=True,
                    boxprops=dict(facecolor='lightblue'),
                    medianprops=dict(color='red', linewidth=2),
                    flierprops=dict(marker='o', color='red', markersize=4))
    axes[i].set_title(col, fontsize=8)

plt.suptitle('Boxplots — Outlier Detection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Count outliers per feature (IQR method)
outlier_counts = {}

for col in df.columns:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_counts[col] = ((df[col] < lower) | (df[col] > upper)).sum()

outlier_series = pd.Series(outlier_counts)
print('Outlier counts per feature:')
outlier_series

In [ ]:
# Apply Winsorization (Cap outliers at fence values)
df_clean = df.copy()

for col in df_clean.columns:
    Q1  = df_clean[col].quantile(0.25)
    Q3  = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)

print('Winsorization applied!')
print('Shape:', df_clean.shape)
df_clean.describe()

---
## Step 5 — Correlation Analysis

**Correlation Range: -1 to +1**
- +1 = Perfect positive (both increase together)
- -1 = Perfect negative (one increases, other decreases)
- 0  = No relationship

**Medical Insight:**
- PSV ↑ + RI ↑ + ColdSpot ↑ = Blockage likely
- BFV ↓ + PA ↓ + HRV ↓ = Blockage likely

In [ ]:
# Correlation matrix
corr_matrix = df_clean.corr()
corr_matrix

In [ ]:
# Heatmap
plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5,
            annot_kws={'size': 7})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top correlated pairs
corr_pairs = corr_matrix.unstack().reset_index()
corr_pairs.columns = ['Feature 1', 'Feature 2', 'Correlation']
corr_pairs = corr_pairs[corr_pairs['Feature 1'] != corr_pairs['Feature 2']]
corr_pairs['Abs Correlation'] = corr_pairs['Correlation'].abs()
corr_pairs = corr_pairs.drop_duplicates(subset=['Abs Correlation'])
corr_pairs.sort_values('Abs Correlation', ascending=False).head(10)

---
## Step 6 — Feature Engineering

**Why Feature Engineering?**
- Original features ek dimension mein hain
- New features = medical domain knowledge encode karna
- Model ko directly blockage patterns dena
- 10 features → 15 features (5 naye medical features)

In [ ]:
# Feature 1: Velocity Ratio
# High PSV + Low BFV = Stenosis (narrow pipe mein pressure high hoti hai)
df_clean['velocity_ratio'] = df_clean['peak_systolic_velocity'] / df_clean['blood_flow_velocity']

# Feature 2: Thermal Stress Index
# High temp diff + Large cold area = Severe ischemia
df_clean['thermal_stress'] = df_clean['temperature_difference'] * df_clean['cold_spot_area_percent']

# Feature 3: Cardiac Load
# High HR + Low PA = Heart overworking due to blockage
df_clean['cardiac_load'] = df_clean['heart_rate'] * df_clean['pulse_amplitude']

# Feature 4: Vascular Resistance Score
# High RI + High PTT = Stiff + Resistant vessel = Blockage
df_clean['vascular_resistance'] = df_clean['resistive_index'] * df_clean['pulse_transit_time']

# Feature 5: Flow Efficiency
# Low BFV / HR = Poor flow per heartbeat = Obstruction
df_clean['flow_efficiency'] = df_clean['blood_flow_velocity'] / df_clean['heart_rate']

print('New features added!')
print('Total features now:', df_clean.shape[1])
df_clean[['velocity_ratio','thermal_stress','cardiac_load','vascular_resistance','flow_efficiency']].describe()

In [ ]:
# Engineered features distribution
eng_feats = ['velocity_ratio', 'thermal_stress', 'cardiac_load', 'vascular_resistance', 'flow_efficiency']

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
colors = ['steelblue', 'orange', 'purple', 'green', 'crimson']

for i, (col, color) in enumerate(zip(eng_feats, colors)):
    axes[i].hist(df_clean[col], bins=30, color=color, edgecolor='black', alpha=0.7)
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=8)
    axes[i].axvline(df_clean[col].mean(), color='red', linestyle='--', linewidth=1.5)

plt.suptitle('Engineered Features Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 7 — Feature Scaling (StandardScaler)

**Why Scaling is important for Deep Learning?**
- PSV = 70-160 cm/s
- RI = 0.55-0.95 (ratio)
- Different scales → Large scale feature dominates training!

**StandardScaler Formula:**
```
X_scaled = (X - mean) / std
Result: mean = 0, std = 1
```

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean.values)

df_scaled = pd.DataFrame(X_scaled, columns=df_clean.columns)
print('Scaled data shape:', df_scaled.shape)
df_scaled.describe()

In [ ]:
# Before vs After scaling comparison (PSV feature)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_clean['peak_systolic_velocity'], bins=30,
             color='tomato', edgecolor='black', alpha=0.7)
axes[0].set_title('PSV — Before Scaling')
axes[0].set_xlabel('cm/s')

axes[1].hist(df_scaled['peak_systolic_velocity'], bins=30,
             color='steelblue', edgecolor='black', alpha=0.7)
axes[1].set_title('PSV — After StandardScaler')
axes[1].set_xlabel('Scaled Value (mean=0, std=1)')

plt.suptitle('Before vs After Scaling', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# All features scaled boxplot
plt.figure(figsize=(14, 5))
df_scaled.boxplot(figsize=(14, 5))
plt.axhline(0, color='red', linestyle='--', linewidth=1)
plt.title('All Features After Scaling (All centered around 0)', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## Step 8 — Save Cleaned Data

In [ ]:
import pickle

# Save clean data
df_clean.to_csv('phase1_clean_data.csv', index=False)

# Save scaled numpy array
np.save('phase1_X_scaled.npy', X_scaled)

# Save scaler for reuse in Phase 2-7
with open('phase1_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save feature names
feature_names = df_clean.columns.tolist()
with open('phase1_feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

print('Files saved:')
print('  phase1_clean_data.csv')
print('  phase1_X_scaled.npy')
print('  phase1_scaler.pkl')
print('  phase1_feature_names.pkl')

---
## Phase 1 Summary

In [ ]:
print('========================================')
print('  PHASE 1 — COMPLETE')
print('========================================')
print(f'Original features  : 10')
print(f'Engineered features: 5')
print(f'Total features     : {df_clean.shape[1]}')
print(f'Total patients     : {df_clean.shape[0]}')
print(f'Missing values     : 0 (median imputed)')
print(f'Outliers           : 0 (winsorized)')
print(f'Scaling            : StandardScaler applied')
print('========================================')
print('  NEXT → PHASE 2: Label Generation')
print('========================================')